# Train AGHV-Net on Oxford 102 Flowers (Colab GPU)

Runs the real training/evaluation code from `github.com/abhilasha-it/AGHV` on a free Colab GPU, so the Streamlit controller can show genuine results instead of the fuzzy rule-only fallback.

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4 is fine).

This trains AGHV-Net plus a ResNet-50 baseline (for the comparison charts) at a reduced epoch count that should fit in a single Colab session. Increase `EPOCHS` below once you've confirmed it runs end-to-end.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/abhilasha-it/AGHV.git
%cd AGHV

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
EPOCHS = 15  # raise this (e.g. 30-50) for a stronger model once the pipeline is confirmed working

## Train + evaluate AGHV-Net

The dataset (Oxford 102 Flowers) downloads automatically on first use into `./data/flowers102`.

In [ ]:
!python -m src.train --model aghv_net --epochs {EPOCHS}

In [ ]:
!python -m src.evaluate --model aghv_net

## Train + evaluate a baseline (for the comparison charts)

Repeat this cell (swap `resnet50` for `vgg16` / `vit_small` / `plain_cnn`) to populate more bars in the Model Comparison tab.

In [ ]:
!python -m src.train --model resnet50 --epochs {EPOCHS}
!python -m src.evaluate --model resnet50

## Get the results out of Colab

`results/checkpoints/aghv_net_best.pt` is typically 150-250 MB (ResNet-50 + ViT-Small) -- too large for a normal GitHub push (GitHub blocks files over 100 MB). Pick **one** of the two options below to host it somewhere the deployed Streamlit app can download it from, then set that URL as the `AGHV_CHECKPOINT_URL` secret in the Streamlit Cloud app settings.

The small `results/metrics/*.json` files (accuracy/precision/recall/F1 + training curves) are tiny -- commit those straight back to the repo (see the last cell) so the comparison charts show real numbers immediately.

### Option A (recommended): Hugging Face Hub

Free, made for hosting model files, gives a stable direct-download URL. Needs a free account at https://huggingface.co and an access token (Settings -> Access Tokens -> create one with **write** scope).

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login, HfApi

login()  # paste your HF token when prompted

REPO_ID = "your-username/aghv-net-flowers102"  # change this
api = HfApi()
api.create_repo(REPO_ID, repo_type="model", exist_ok=True)
api.upload_file(
    path_or_fileobj="results/checkpoints/aghv_net_best.pt",
    path_in_repo="aghv_net_best.pt",
    repo_id=REPO_ID,
    repo_type="model",
)
print(f"Checkpoint URL: https://huggingface.co/{REPO_ID}/resolve/main/aghv_net_best.pt")

### Option B: Google Drive

Simpler if you'd rather not create a Hugging Face account, but the direct-download link is a bit more fragile (Drive sometimes shows a virus-scan warning page for large files instead of downloading directly).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp results/checkpoints/aghv_net_best.pt /content/drive/MyDrive/aghv_net_best.pt
print("Uploaded to Google Drive. Right-click the file there -> Share -> Anyone with the link, "
      "copy the file ID from the share URL, and use https://drive.google.com/uc?export=download&id=FILE_ID "
      "as the AGHV_CHECKPOINT_URL secret.")

## Commit the metrics JSON back to the repo

These are small text files -- safe to push directly. Requires a GitHub personal access token (github.com -> Settings -> Developer settings -> Personal access tokens) with `repo` scope.

In [ ]:
import getpass

token = getpass.getpass("GitHub personal access token: ")
!git config user.email "you@example.com"
!git config user.name "Colab Training Run"
!git add results/metrics/
!git commit -m "Add real training/evaluation metrics from Colab run"
!git push https://{token}@github.com/abhilasha-it/AGHV.git master